# Part 02 — Known & Unknown pIC50: Molecular Descriptor & Fingerprint Studio

**Google Colab edition · Native CSV upload · high-contrast interface**

This standalone Colab notebook preserves the supplied 76-descriptor scientific engine, the installed RDKit full descriptor registry and **ECFP4 (Morgan radius 2, 2048 bits)**. You can select one, two or all three descriptor sets.

**Run all cells:** Colab opens the official native CSV file picker in **Cell 3**. Select a CSV with compound names and original SMILES (optional `pIC50`), and the high-contrast dashboard in **Cell 4** loads it automatically. No paths or Drive links are required.

Developed by **Md Fazlay Rabbi**, Department of Zoology, University of Rajshahi · **rabbi.ms244@gmail.com**.


In [1]:
# @title  CELL 1 — Automatic dependency check (install missing packages only)
import importlib.util, subprocess, sys
packages = {"rdkit":"rdkit", "pandas":"pandas", "numpy":"numpy",
            "matplotlib":"matplotlib", "ipywidgets":"ipywidgets"}
missing=[pkg for module,pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing only missing packages:", ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])
else:
    print("All required packages found. No installations needed.")


Installing only missing packages: rdkit


In [2]:
# @title Cell 2
"""Part 2 | Known & Unknown pIC50 — Descriptor generation scientific engine.
For Google Colab: copied inline into notebook cell 2.
"""
from __future__ import annotations
import csv, json, math, os, re, shutil, tempfile, time, zipfile
from collections import Counter
from pathlib import Path
from datetime import datetime, timezone
from typing import Callable
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, rdBase, RDLogger
from rdkit.Chem import Descriptors, AllChem, Fragments, QED, rdMolDescriptors, rdFingerprintGenerator
from rdkit.Chem import rdmolops
RDLogger.DisableLog('rdApp.warning')

TOPO_NAMES = ['WienerIndex','ZagrebIndex1','ZagrebIndex2','BalabanJ','TPSA',
'Kappa1','Kappa2','Kappa3','Chi0','Chi1','Chi0n','Chi1n','Chi2n','Chi3n','Chi4n','HallKierAlpha','PetitjeanNumber']
PHYS_NAMES = ['MolecularWeight','ExactMolWt','LogP','LogD','NumHDonors','NumHAcceptors','NumRotatableBonds',
'MolMR','LabuteASA','FractionCsp3','NumHeteroatoms','NumSaturatedCarbocycles',
'NumSaturatedHeterocycles','NumAliphaticCarbocycles','NumAliphaticHeterocycles',
'NumAromaticCarbocycles','NumAromaticHeterocycles','RingCount','NumRings','NumAromaticRings',
'MaxPartialCharge','MinPartialCharge','MaxAbsPartialCharge','MinAbsPartialCharge']
THREED_NAMES = ['RadiusOfGyration','Asphericity','Eccentricity','InertialShapeFactor','NPR1','NPR2',
'SpherocityIndex','PMI1','PMI2','PMI3']
FRAGMENT_NAMES = ['fr_Al_COO','fr_Al_OH','fr_Ar_COO','fr_Ar_OH','fr_COO','fr_benzene',
'fr_NH0','fr_NH1','fr_NH2','fr_N_O','fr_Ndealkylation1','fr_Ndealkylation2',
'fr_amide','fr_aniline','fr_phenol']
QED_NAMES = ['QED','QED_MW','QED_ALOGP','QED_HBA','QED_HBD','QED_PSA','QED_ROTB','QED_AROM','QED_ALERTS','LipinskiViolations']
LEGACY76_NAMES = TOPO_NAMES + PHYS_NAMES + THREED_NAMES + FRAGMENT_NAMES + QED_NAMES
assert len(LEGACY76_NAMES) == 76 and len(set(LEGACY76_NAMES)) == 76
RDKIT_REGISTRY = list(Descriptors._descList)
RD_NAMES = [name for name,func in RDKIT_REGISTRY]
if len(RD_NAMES) != len(set(RD_NAMES)):
    raise RuntimeError('RDKit descriptor registry contains duplicate names')
ECFP_NAMES = [f'ECFP4_{i:04d}' for i in range(2048)]
FEATURES = {'76': LEGACY76_NAMES, 'rdkit': RD_NAMES, 'ecfp': ECFP_NAMES}

# Known identity-of-definition aliases; avoid statistical deduplication before splitting
# data for QSAR model training. Standalone output files still keep every requested column.
DEFINITION_ALIASES = {
    'MolecularWeight': 'MolWt',
    'LogP': 'MolLogP',
    'FractionCsp3': 'FractionCSP3',
    'NumRings': 'RingCount',
    'QED': 'qed',
}

def plan_definition_deduplication(selected, enabled=True):
    """Deterministic pre-ML descriptor-identity filtering, never fit on molecule values.

    Keeps the first family in the user's selection order (76 -> RDKit -> ECFP4),
    avoiding RDKit descriptors that repeat custom76 properties. No data leakage.
    """
    keep=[]
    dropped=[]
    seen={}
    for family in selected:
        for name in FEATURES[family]:
            col=('D76_' if family=='76' else 'RDKit_' if family=='rdkit' else '')+name
            canonical=DEFINITION_ALIASES.get(name,name) if family!='ecfp' else 'ECFP4_'+name
            if enabled and family=='76' and name=='LogD':
                dropped.append({'removed_column':col,'retained_column':'',
                                'reason':'LogD is undefined without a validated pH-dependent model; the raw 76-feature CSV retains NaN.'})
            elif enabled and canonical in seen:
                dropped.append({'removed_column':col,'retained_column':seen[canonical],
                                'reason':'Known identical descriptor definition or explicitly verified alias.'})
            else:
                keep.append(col)
                seen[canonical]=col
    return keep,dropped


def safe_float(x):
    try:
        value=float(x)
        return value if math.isfinite(value) else np.nan
    except (TypeError,ValueError,OverflowError):
        return np.nan


def _name_func(mol, key):
    func = getattr(Descriptors, key, None)
    if not callable(func):
        return np.nan
    try:
        return safe_float(func(mol))
    except Exception:
        return np.nan


def legacy76(mol):
    """Reproduces the names from user's original first notebook.
    Fixed: LogD is undefined from structure alone, so kept as NaN (not mislabeled LogP).
    Missing fragment implementations are NaN rather than a false zero.
    """
    d = dict.fromkeys(LEGACY76_NAMES, np.nan)
    # Topological descriptors: built-in RDKit + exact custom graph metrics from legacy code
    for name in ('BalabanJ','TPSA','Kappa1','Kappa2','Kappa3','Chi0','Chi1','Chi0n','Chi1n',
                 'Chi2n','Chi3n','Chi4n','HallKierAlpha'):
        d[name] = _name_func(mol, name)
    try:
        distances=np.asarray(rdmolops.GetDistanceMatrix(mol),dtype=float)
        n=distances.shape[0]
        # RDKit uses a large off-path value for disconnected atoms; do not infer finite Wiener.
        if n > 0 and np.isfinite(distances).all() and np.max(distances) < 1e6:
            d['WienerIndex']=float(distances[np.triu_indices(n, k=1)].sum())
            eccentricity=distances.max(axis=1)
            diameter=float(eccentricity.max())
            radius=float(eccentricity.min())
            d['PetitjeanNumber']=(diameter-radius)/diameter if diameter>0 else 0.0
        adj=np.asarray(rdmolops.GetAdjacencyMatrix(mol), dtype=float)
        degrees=adj.sum(axis=1)
        d['ZagrebIndex1']=float(np.sum(degrees**2))
        d['ZagrebIndex2']=float(sum(degrees[i]*degrees[j] for i,j in zip(*np.where(np.triu(adj,1)>0))))
    except Exception:
        pass

    orig_phys={
        'MolecularWeight':'MolWt', 'ExactMolWt':'ExactMolWt','LogP':'MolLogP',
        'NumHDonors':'NumHDonors','NumHAcceptors':'NumHAcceptors',
        'NumRotatableBonds':'NumRotatableBonds','MolMR':'MolMR','LabuteASA':'LabuteASA',
        'FractionCsp3':'FractionCSP3','NumHeteroatoms':'NumHeteroatoms',
        'NumSaturatedCarbocycles':'NumSaturatedCarbocycles',
        'NumSaturatedHeterocycles':'NumSaturatedHeterocycles',
        'NumAliphaticCarbocycles':'NumAliphaticCarbocycles',
        'NumAliphaticHeterocycles':'NumAliphaticHeterocycles',
        'NumAromaticCarbocycles':'NumAromaticCarbocycles',
        'NumAromaticHeterocycles':'NumAromaticHeterocycles',
        'RingCount':'RingCount','NumRings':'RingCount','NumAromaticRings':'NumAromaticRings',
        'MaxPartialCharge':'MaxPartialCharge','MinPartialCharge':'MinPartialCharge',
        'MaxAbsPartialCharge':'MaxAbsPartialCharge','MinAbsPartialCharge':'MinAbsPartialCharge',
    }
    for outname,rdname in orig_phys.items():
        d[outname]=_name_func(mol,rdname)
    d['LogD'] = np.nan  # Requires specified ionization/pH model; old notebook mislabeled LogP as LogD.
    # Physicochemical 3D descriptors: reproducible ETKDGv3 3D embedding, MMFF if available.
    try:
        mol3d=Chem.AddHs(Chem.Mol(mol))
        params=AllChem.ETKDGv3()
        params.randomSeed=42
        params.maxIterations=30
        if AllChem.EmbedMolecule(mol3d, params)==0:
            if AllChem.MMFFHasAllMoleculeParams(mol3d):
                try: AllChem.MMFFOptimizeMolecule(mol3d, maxIters=200)
                except Exception: pass
            geometry={
                'RadiusOfGyration':'CalcRadiusOfGyration', 'Asphericity':'CalcAsphericity',
                'Eccentricity':'CalcEccentricity','InertialShapeFactor':'CalcInertialShapeFactor',
                'NPR1':'CalcNPR1','NPR2':'CalcNPR2','SpherocityIndex':'CalcSpherocityIndex',
                'PMI1':'CalcPMI1','PMI2':'CalcPMI2','PMI3':'CalcPMI3',
            }
            for name,fn in geometry.items():
                try: d[name]=safe_float(getattr(rdMolDescriptors,fn)(mol3d))
                except Exception: d[name]=np.nan
    except Exception:
        pass
    for name in FRAGMENT_NAMES:
        fn=getattr(Fragments,name,None)
        if callable(fn):
            try: d[name]=safe_float(fn(mol))
            except Exception: pass
    try:
        q=QED.qed(mol)
        p=QED.properties(mol)
        d['QED']=safe_float(q)
        for field in ('MW','ALOGP','HBA','HBD','PSA','ROTB','AROM','ALERTS'):
            d['QED_'+field]=safe_float(getattr(p,field))
        d['LipinskiViolations']=int(p.MW>500)+int(p.ALOGP>5)+int(p.HBD>5)+int(p.HBA>10)
    except Exception:
        pass
    return [d[n] for n in LEGACY76_NAMES]


def rdkit_all(mol):
    values=[]
    for name,fn in RDKIT_REGISTRY:
        try: values.append(safe_float(fn(mol)))
        except Exception: values.append(np.nan)
    return values

FPGEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=False)
def fp_vec(mol):
    fp = FPGEN.GetFingerprint(mol)
    arr = np.zeros(2048,dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp,arr)
    return arr,fp


def auto_detect(columns,kind):
    preferences={
       'smiles':['smiles','canonical_smiles','isomeric_smiles','original_smiles','SMILES','Smiles'],
       'name':['compound_name','molecule_chembl_id','compound','name','Compound_Name','id','ID'],
       'pic50':['pIC50','pic50','p_ic50','pIC50_value','PIC50'],
    }[kind]
    for q in preferences:
        for c in columns:
            if c.casefold()==q.casefold(): return c
    return None


def safe_prefix(s):
    return re.sub(r'[^A-Za-z0-9_\-]+','_',str(s)).strip('_')[:60] or 'QSAR'


def generate_tables(df:pd.DataFrame, *, name_col:str, smiles_col:str,
                    activity_col:str|None, selected:list[str], output_root:str|Path,
                    prefix:str='KnownUnknown', qc_plots:bool=True, deduplicate:bool=True,
                    progress:Callable[[int,int],None]|None=None,
                    log:Callable[[str],None]|None=None):
    """Streaming CSV writer to minimize peak memory with 2048-bit fingerprints.
    Keeps original metadata, original SMILES and input row order. Never imputes/scales data.
    """
    if not selected or any(k not in FEATURES for k in selected):
        raise ValueError('Choose one or more descriptor families')
    if name_col not in df.columns or smiles_col not in df.columns:
        raise ValueError('Select valid compound name and SMILES columns')
    if name_col==smiles_col: raise ValueError('Compound name and SMILES columns must be different')
    if activity_col and activity_col not in df.columns: raise ValueError('Activity column unavailable')
    names={s for s in selected}
    prefix=safe_prefix(prefix)
    folder=Path(output_root)
    folder.mkdir(parents=True,exist_ok=True)
    # Preserves source columns incl compound IDs, optional activity, plus status.
    source_cols=list(map(str,df.columns))
    if len(source_cols)!=len(set(source_cols)):
        raise ValueError('Duplicate CSV column names are not supported; rename duplicated headers')
    metas=['source_row','compound_name','original_smiles','pIC50_status'] + [f'source_{c}' for c in source_cols]
    filename_for={'76':f'{prefix}_76_Descriptors.csv',
                  'rdkit':f'{prefix}_RDKit_All_{len(RD_NAMES)}.csv',
                  'ecfp':f'{prefix}_ECFP4_2048.csv'}
    outpaths={k:folder/filename_for[k] for k in selected}
    combined=folder/f'{prefix}_Selected_Combined_RAW.csv'
    cleaned=folder/f'{prefix}_Selected_Combined_DEDUPLICATED.csv'
    dedup_audit=folder/f'{prefix}_Descriptor_Deduplication_Audit.csv'
    clean_names, dropped_columns=plan_definition_deduplication(selected,deduplicate)
    all_names=[('D76_' if key=='76' else 'RDKit_' if key=='rdkit' else '')+n for key in selected for n in FEATURES[key]]
    clean_indices=[i for i,name in enumerate(all_names) if name in set(clean_names)]
    invalid=folder/f'{prefix}_Invalid_SMILES.csv'
    status_count=Counter()
    failed=[]
    preview=[]
    qc_rd=[]; qc_fp=[]
    start=time.monotonic()
    # Open output streams safely with one pass; all selected families and combined in lockstep.
    import contextlib
    with contextlib.ExitStack() as stack:
        writers={}
        for key,path in outpaths.items():
            f=stack.enter_context(open(path,'w',newline='',encoding='utf-8-sig'))
            w=csv.writer(f)
            w.writerow(metas+[('D76_' if key=='76' else 'RDKit_' if key=='rdkit' else '')+n for n in FEATURES[key]])
            writers[key]=w
        cf=stack.enter_context(open(combined,'w',newline='',encoding='utf-8-sig'))
        cw=csv.writer(cf)
        cw.writerow(metas+all_names)
        clf=stack.enter_context(open(cleaned,'w',newline='',encoding='utf-8-sig'))
        clw=csv.writer(clf)
        clw.writerow(metas+clean_names)
        n=len(df)
        for seq,record in enumerate(df.itertuples(index=False,name=None),start=1):
            row=dict(zip(source_cols,record))
            raw_smi=row.get(smiles_col)
            smi='' if pd.isna(raw_smi) else str(raw_smi).strip()
            cname='' if pd.isna(row[name_col]) else str(row[name_col]).strip()
            try: mol=Chem.MolFromSmiles(smi) if smi else None
            except Exception: mol=None
            if mol is None:
                failed.append({'source_row':seq,'compound_name':cname,'original_smiles':smi,'reason':'Invalid or empty SMILES'})
                status_count['invalid']+=1
            else:
                activity=row.get(activity_col) if activity_col else None
                status='known' if activity_col and pd.notna(pd.to_numeric(activity,errors='coerce')) else 'unknown'
                status_count[status]+=1
                meta=[seq,cname,smi,status]+[row.get(c) for c in source_cols]
                calculated={}
                for k in selected:
                    if k=='76': calculated[k]=legacy76(mol)
                    elif k=='rdkit': calculated[k]=rdkit_all(mol)
                    else:
                        arr, fp=fp_vec(mol)
                        calculated[k]=arr.tolist()
                        if qc_plots and len(qc_fp)<200: qc_fp.append(fp)
                    writers[k].writerow(meta+calculated[k])
                flat=[v for k in selected for v in calculated[k]]
                cw.writerow(meta+flat)
                clw.writerow(meta+[flat[i] for i in clean_indices])
                if qc_plots and 'rdkit' in names and len(qc_rd)<300:
                    qc_rd.append(calculated['rdkit'])
                if len(preview)<5: preview.append({'compound_name':cname,'smiles':smi,'pIC50_status':status})
            if progress and (seq%max(1,min(50,n//100 or 1))==0 or seq==n):
                progress(seq,n)
    pd.DataFrame(failed,columns=['source_row','compound_name','original_smiles','reason']).to_csv(invalid,index=False)
    pd.DataFrame(dropped_columns,columns=['removed_column','retained_column','reason']).to_csv(dedup_audit,index=False)
    for k in selected:
        p=folder/f'{prefix}_{k}_feature_names.txt'
        p.write_text('\n'.join(FEATURES[k])+'\n',encoding='utf8')
    if qc_plots: write_qc_figures(folder,prefix,qc_rd,qc_fp,log)
    n_valid=sum(status_count[x] for x in ('known','unknown'))
    methods=(
        'PART 2 — KNOWN AND UNKNOWN pIC50 COMPOUND DESCRIPTOR GENERATION\n'
        'Molecular structures were parsed from the uploaded SMILES using RDKit version '
        f'{rdBase.rdkitVersion}. Exactly {len(LEGACY76_NAMES)} legacy-named properties, '
        f'{len(RD_NAMES)} RDKit registry descriptors and/or 2048 ECFP4 bits (Morgan radius 2) '
        'were exported according to user selection. Original family tables and an unmodified combined table were exported. '
        'Input compound identifiers, original SMILES, original row indices and optional pIC50 values were retained. No imputation, normalization '
        'or supervised feature selection was applied. Failed SMILES were excluded and logged.\n'
        'An optional deduplicated combined file removes only known identical descriptor definitions and the undefined LogD column. '
        'It does NOT remove features using sample-wide correlation or variance, which must be learned in training folds. '
        'The audit CSV lists all removed features and retained aliases.\n'
        '76-feature notes: ten molecular geometry descriptors rely on one ETKDGv3 conformer '
        '(seed 42), optionally MMFF optimized; geometry failures remain missing. Legacy LogD '
        'is deliberately missing (NaN) because the original script used logP as an invalid '
        'pH-independent LogD surrogate. RingCount and NumRings remain redundant for legacy compatibility.\n'
        'ECFP4 uses radius=2, bit length=2048, no chirality encoding to match the original notebook.\n'
    )
    (folder/'Methods.txt').write_text(methods,encoding='utf8')
    summary={
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'rdkit_version':rdBase.rdkitVersion,'rdkit_registry_count':len(RD_NAMES),
        'selected_families':selected,'descriptor_features':{k:len(FEATURES[k]) for k in selected},
        'input_rows':len(df),'valid_rows':n_valid,'invalid_rows':len(failed),
        'known_pIC50':status_count['known'],'unknown_pIC50':status_count['unknown'],
        'input_columns':source_cols,'mapped_columns':{'compound_name':name_col,'smiles':smiles_col,'pIC50':activity_col},
        'qc_plots':bool(qc_plots),'elapsed_seconds':round(time.monotonic()-start,2),
        'definition_deduplication_enabled':bool(deduplicate),
        'raw_feature_count':len(all_names), 'cleaned_feature_count':len(clean_names),
        'removed_definition_duplicate_count':len(dropped_columns),
        'limitations':['A 3D conformer is not an ensemble or experimentally measured conformation.',
                       'LogD cannot be inferred solely by reusing logP. Legacy 76 LogD exported as NaN.',
                       'RDKit descriptor count can change with installed version.']
    }
    (folder/'Summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf8')
    (folder/'Results.txt').write_text(
        f"Input compounds: {len(df)}\nSuccessfully processed: {n_valid}\n"
        f"Known pIC50: {status_count['known']}\nUnknown pIC50: {status_count['unknown']}\n"
        f"Invalid/empty SMILES: {len(failed)}\n"
        f"Raw selected features: {len(all_names)}\n"
        f"Cleaned selected features: {len(clean_names)}\n"
        f"Features removed by definition audit: {len(dropped_columns)}\n"+
        ''.join(f'{key} features: {len(FEATURES[key])}\n' for key in selected),encoding='utf8')
    zpath=folder.parent/f'{folder.name}.zip'
    with zipfile.ZipFile(zpath,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
        for path in sorted(folder.iterdir()):
            if path.is_file(): zf.write(path,arcname=path.name)
    if log:log(f'Completed: {n_valid}/{len(df)} valid; output: {zpath}')
    return {'summary':summary,'files':{k:str(p) for k,p in outpaths.items()},
            'combined':str(combined),'cleaned':str(cleaned),'audit':str(dedup_audit),'zip':str(zpath),'folder':str(folder),'preview':preview}


def write_qc_figures(folder,prefix,rd_rows,fp_objects,log=None):
    """Optional descriptive QC only, sampled; no activity leakage or modeling."""
    if len(rd_rows)>=3:
        import matplotlib.pyplot as plt
        matrix=pd.DataFrame(rd_rows,columns=RD_NAMES)
        corr=matrix.corr(numeric_only=True).abs()
        corr.to_csv(folder/f'{prefix}_RDKit_sample_descriptor_correlations.csv')
        arr=corr.to_numpy(copy=True)
        np.fill_diagonal(arr,np.nan)
        scores=np.nan_to_num(arr,nan=0).max(axis=0)
        ix=np.argsort(scores)[-min(18,len(scores)):]
        sub=corr.iloc[ix,ix]
        fig,ax=plt.subplots(figsize=(11,9))
        im=ax.imshow(sub.to_numpy(),vmin=0,vmax=1,cmap='viridis')
        ax.set_xticks(range(len(ix)),labels=sub.columns,rotation=90,fontsize=8,fontweight='bold')
        ax.set_yticks(range(len(ix)),labels=sub.index,fontsize=8,fontweight='bold')
        ax.set_title('RDKit descriptor absolute correlations (QC sample)',fontweight='bold')
        fig.colorbar(im,ax=ax,label='Absolute Pearson correlation')
        fig.tight_layout()
        fig.savefig(folder/f'{prefix}_Descriptor_Correlations.tiff',dpi=600)
        plt.close(fig)
        if log:log('Saved sampled RDKit descriptor correlation figure')
    if len(fp_objects)>=3:
        import matplotlib.pyplot as plt
        sims=[]
        for i in range(1,len(fp_objects)):
            sims.extend(DataStructs.BulkTanimotoSimilarity(fp_objects[i],fp_objects[:i]))
        fig,ax=plt.subplots(figsize=(7.5,4.8))
        ax.hist(sims,bins=25,alpha=0.85,edgecolor='black',color='#2563eb')
        ax.set_title('ECFP4 Tanimoto Similarity (QC sample)',fontweight='bold')
        ax.set_xlabel('Tanimoto similarity',fontweight='bold')
        ax.set_ylabel('Pair frequency',fontweight='bold')
        for labels in (ax.get_xticklabels(),ax.get_yticklabels()):
            for t in labels:t.set_fontweight('bold')
        fig.tight_layout()
        fig.savefig(folder/f'{prefix}_Fingerprint_Diversity.tiff',dpi=600)
        plt.close(fig)
        if log:log('Saved sampled ECFP4 Tanimoto figure')


In [3]:
# @title CELL 3 — NATIVE GOOGLE COLAB CSV UPLOAD (NO PATH ENTRY)
# Google's documented `files.upload()` runs in this ordinary cell.
import io
import pandas as pd
from IPython.display import display, HTML

QSAR_INPUT_DF = None
QSAR_INPUT_NAME = None

display(HTML('''
<div style="max-width:1120px;margin:8px auto 14px;padding:22px 26px;background:linear-gradient(110deg,#102d52,#136b82);border-radius:16px;color:#ffffff;font-family:Arial,sans-serif">
  <div style="font-size:12px;letter-spacing:1.6px;font-weight:800;color:#d1edf7">PART 02 · FILE IMPORT</div>
  <div style="font-size:25px;font-weight:900;margin-top:7px">Select your compound CSV</div>
  <div style="font-size:14px;color:#e3f1f8;margin-top:7px">Use the browser file picker below. No file path or Google Drive setup required.</div>
</div>
'''))
try:
    from google.colab import files as colab_files
    uploaded_files = colab_files.upload()
    if uploaded_files:
        QSAR_INPUT_NAME, binary_contents = next(iter(uploaded_files.items()))
        if not QSAR_INPUT_NAME.lower().endswith('.csv'):
            raise ValueError('Please upload a .csv file, not another format.')
        try:
            QSAR_INPUT_DF = pd.read_csv(io.BytesIO(binary_contents), encoding='utf-8-sig', low_memory=False)
        except UnicodeDecodeError:
            QSAR_INPUT_DF = pd.read_csv(io.BytesIO(binary_contents), encoding='latin-1', low_memory=False)
        if QSAR_INPUT_DF.empty:
            raise ValueError('The uploaded CSV contains no data rows.')
        print(f'UPLOAD SUCCESSFUL: {QSAR_INPUT_NAME} | {len(QSAR_INPUT_DF):,} compounds | {len(QSAR_INPUT_DF.columns)} columns')
        display(QSAR_INPUT_DF.head(3))
    else:
        print('No CSV selected. You can rerun this cell to upload, or use the backup uploader in the dashboard.')
except ImportError:
    print('This cell uses the native Google Colab uploader. Open this notebook at colab.research.google.com.')
    print('A backup ipywidgets uploader is also available in the dashboard (next cell).')
except Exception as exc:
    print(f'UPLOAD ERROR: {type(exc).__name__}: {exc}')
    print('Rerun this cell and choose a valid CSV.')


Saving Final_compound.csv to Final_compound.csv
UPLOAD SUCCESSFUL: Final_compound.csv | 1,913 compounds | 2 columns


,compound_name,smiles
0,IMPHY002338,OCC1=C(C)[C@@H]([C@@H](OC1=O)[C@H]([C@@]1(O)CC...
1,IMPHY002349,O=C1O[C@H]([C@H](C(=C1C)C)O)[C@H]([C@H]1CC[C@@...
2,IMPHY003143,CC1=C(C)C(=O)O[C@H](C1)[C@@]([C@@]1(O)C[C@H]2[...


In [4]:
# @title CELL 4 — High-contrast Colab dashboard for Part 2 descriptor generation
import io, os, time, traceback, html
from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, FileLink
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

STYLES = r'''
<style>
.qs-app{font-family:Arial,Helvetica,sans-serif!important;max-width:1150px;color:#132f48!important}
.qs-hero{color:#fff!important;background:linear-gradient(115deg,#102b4c,#195782 60%,#0a8190);border-radius:16px;padding:24px 28px;margin:8px 0 15px;box-shadow:0 8px 20px #12314b22}
.qs-kicker{color:#d5edf6!important;letter-spacing:1.3px;font-size:12px;font-weight:800}
.qs-heading{color:#fff!important;font-size:27px;font-weight:900;line-height:1.15;margin:8px 0}
.qs-tagline{color:#e7f4ff!important;font-size:14px}
.qs-byline{color:#fff!important;font-size:12px;font-weight:750;margin-top:11px}
.qs-guide{background:#eaf5ff;color:#183753!important;border:1px solid #bfd8ec;border-radius:12px;padding:14px 18px;line-height:1.75;margin-bottom:13px;font-size:13px}
.qs-guide b{color:#143c5d!important}
.qs-panel{background:#fff!important;color:#132f48!important;border:1px solid #d6e4f1!important;border-radius:13px!important;box-shadow:0 2px 8px #08284313;margin:8px 0 11px!important; padding:17px 21px!important}
.qs-panel .widget-label, .qs-panel .widget-checkbox label, .qs-panel label,
.qs-panel .widget-readout, .qs-panel .widget-html-content,
.qs-panel .widget-checkbox .widget-label, .qs-panel .widget-text .widget-label,
.qs-panel .widget-dropdown .widget-label,
.qs-panel .widget-progress .widget-label{color:#163852!important;font-size:13px!important;font-weight:650!important}
.qs-panel .widget-text input,.qs-panel .widget-dropdown select,
.qs-panel .widget-combobox input{background-color:#ffffff!important;color:#112d48!important;border:1px solid #a8c3d8!important;border-radius:7px!important;font-size:14px!important;font-weight:600!important}
.qs-panel .widget-button{border-radius:8px!important;font-weight:800!important;font-size:13px!important}
.qs-panel .widget-checkbox input{accent-color:#1767a4!important}
.qs-primary button{background:#088776!important;color:#ffffff!important}
.qs-download button{background:#236bb1!important;color:#ffffff!important}
.qs-backup button{background:#e6f3fa!important;color:#194d73!important}
.qs-panel .widget-output{color:#143d58!important}
.qs-app .widget-html{color:#143d58!important}
</style>
'''
display(HTML(STYLES))

def escaped(x):
    return html.escape(str(x), quote=True)

def header(text, subtitle=''):
    return widgets.HTML(value=(
        f'<div style="font-family:Arial;color:#113e5f;font-size:19px;font-weight:900;margin:0 0 4px">{text}</div>'
        f'<div style="font-family:Arial;color:#486783;font-size:13px;line-height:1.5;margin:0 0 8px">{subtitle}</div>'
    ))

def field(label, control, width=None):
    if width:
        control.layout.width=width
    return widgets.VBox([
        widgets.HTML(value=f'<b style="color:#173957;font-size:13px;font-family:Arial">{label}</b>'),
        control
    ],layout=widgets.Layout(min_width='200px',margin='2px 16px 6px 0'))

def panel(*items):
    box=widgets.VBox(items, layout=widgets.Layout(width='100%'))
    box.add_class('qs-panel')
    return box

def checkline(box, label, note=''):
    box.description=''
    box.indent=False
    box.layout=widgets.Layout(width='30px',min_width='30px',margin='0 9px 0 0')
    title=(f'<span style="color:#183954;font-weight:750;font-size:14px;">{label}</span>'
           +(f'<span style="color:#5a7188;font-size:12px"> — {note}</span>' if note else ''))
    return widgets.HBox([box, widgets.HTML(value=title)],layout=widgets.Layout(align_items='center',margin='6px 0'))

display(HTML('''
<div class="qs-app">
<div class="qs-hero">
 <div class="qs-kicker">PART 02 · KNOWN &amp; UNKNOWN pIC50</div>
 <div class="qs-heading">Molecular Descriptor &amp; Fingerprint Studio</div>
 <div class="qs-tagline">76 original features &nbsp;·&nbsp; Complete RDKit descriptors &nbsp;·&nbsp; ECFP4 (Morgan radius 2, 2048 bits)</div>
 <div class="qs-byline">Developed by Md Fazlay Rabbi &nbsp;|&nbsp; Department of Zoology, University of Rajshahi &nbsp;|&nbsp; rabbi.ms244@gmail.com</div>
</div>
<div class="qs-guide"><b>QUICK USER GUIDE</b> &nbsp; 1. Upload your CSV in <b>Cell 3</b> using the native Colab picker.
&nbsp; 2. Confirm compound_name / smiles / optional pIC50 columns below.
&nbsp; 3. Select one, two or all three feature sets.
&nbsp; 4. Generate descriptors and review the progress.
&nbsp; 5. Download your chosen CSV or the full ZIP.
<br><b>Scientific note:</b> The 76-feature family contains ten 3D descriptors. Undefined legacy LogD stays missing rather than being replaced with logP.</div>
</div>'''))

state={'frame':None,'filename':None,'result':None}
status=widgets.HTML('<b style="color:#516d83">No CSV loaded. Run the native upload cell above.</b>')
preview_out=widgets.Output(layout=widgets.Layout(max_height='185px',overflow='auto',width='100%'))

# Backup uploader: only a CSV file widget; there is NO path text field.
uploader=widgets.FileUpload(accept='.csv',multiple=False,description='Choose CSV (backup)',layout=widgets.Layout(width='230px'))
load_uploaded_button=widgets.Button(description='Use Selected CSV',icon='check',layout=widgets.Layout(width='190px',height='36px'))
load_uploaded_button.add_class('qs-backup')

smiles_select=widgets.Dropdown(options=[('Upload a CSV first','')],description='',layout=widgets.Layout(width='295px'))
name_select=widgets.Dropdown(options=[('Upload a CSV first','')],description='',layout=widgets.Layout(width='295px'))
activity_select=widgets.Dropdown(options=[('No pIC50 column','')],description='',layout=widgets.Layout(width='295px'))

select76=widgets.Checkbox(value=True)
select_rd=widgets.Checkbox(value=True)
select_fp=widgets.Checkbox(value=True)
qc_box=widgets.Checkbox(value=True)
dedup_box=widgets.Checkbox(value=True)
output_prefix=widgets.Text(value='Known_Unknown_pIC50',description='',layout=widgets.Layout(width='340px'))
run_btn=widgets.Button(description='Generate Selected Descriptors',icon='play',layout=widgets.Layout(width='310px',height='43px'))
run_btn.button_style='success';run_btn.add_class('qs-primary')
progressbar=widgets.IntProgress(value=0,min=0,max=100,description='',bar_style='info',layout=widgets.Layout(width='100%'))
progress_text=widgets.HTML('<b style="color:#173956">0 / 0 compounds</b>')
log_output=widgets.Output(layout=widgets.Layout(max_height='235px',overflow='auto',border='1px solid #d7e4f1',padding='10px',width='100%'))
summary_out=widgets.Output(layout=widgets.Layout(max_height='290px',overflow='auto',width='100%'))
download_options=widgets.Dropdown(options=[('Run an analysis first','')],description='',layout=widgets.Layout(width='405px'))
download_btn=widgets.Button(description='Download Selected',icon='download',layout=widgets.Layout(width='190px',height='38px'))
download_btn.button_style='primary';download_btn.add_class('qs-download')
download_zip_btn=widgets.Button(description='Download Complete ZIP',icon='download',layout=widgets.Layout(width='230px',height='38px'))
download_zip_btn.button_style='primary';download_zip_btn.add_class('qs-download')
download_status=widgets.Output(layout=widgets.Layout(max_height='130px'))

def _notify(msg):
    status.value = msg

def _load_dataframe(df,filename):
    if not isinstance(df,pd.DataFrame) or df.empty:
        raise ValueError('The selected CSV contains no data rows.')
    if df.columns.duplicated().any():
        raise ValueError('Duplicate CSV column names were detected. Please make them unique.')
    state.update(frame=df,filename=str(filename),result=None)
    cols=list(df.columns)
    for w in (name_select,smiles_select):
        w.options=[(str(c),c) for c in cols]
    activity_select.options=[('No pIC50 column','')]+[(str(c),c) for c in cols]
    sm=auto_detect(cols,'smiles'); nm=auto_detect(cols,'name'); pc=auto_detect(cols,'pic50')
    if sm: smiles_select.value=sm
    elif len(cols)>1: smiles_select.value=cols[1]
    if nm:name_select.value=nm
    else:name_select.value=cols[0]
    activity_select.value=pc if pc else ''
    _notify(f'<b style="color:#087966">CSV loaded successfully: {escaped(filename)} — {len(df):,} compounds, {len(cols)} columns.</b>')
    with preview_out:
        preview_out.clear_output(wait=True)
        display(df.head(5))
    download_options.options=[('Run an analysis first','')]

def _parse_uploaded_value(files):
    if not files:
        return None
    if isinstance(files,dict):
        fname,entry=next(iter(files.items()))
        content=entry.get('content') if isinstance(entry,dict) else entry['content']
    else:
        entry=files[0]
        fname=entry.get('name') if isinstance(entry,dict) else entry['name']
        content=entry.get('content') if isinstance(entry,dict) else entry['content']
    raw=content.tobytes() if hasattr(content,'tobytes') else bytes(content)
    try:
        df=pd.read_csv(io.BytesIO(raw),encoding='utf-8-sig',low_memory=False)
    except UnicodeDecodeError:
        df=pd.read_csv(io.BytesIO(raw),encoding='latin-1',low_memory=False)
    return df,fname

def on_upload_change(change):
    try:
        result=_parse_uploaded_value(change['new'])
        if result is not None:
            _load_dataframe(*result)
    except Exception as exc:
        _notify(f'<b style="color:#ad2330">File upload error: {escaped(exc)}</b>')

def on_upload_fallback(_):
    try:
        result=_parse_uploaded_value(uploader.value)
        if result is None:
            _notify('<b style="color:#ad2330">First select a CSV in the backup uploader, or rerun Cell 3.</b>')
        else:
            _load_dataframe(*result)
    except Exception as exc:
        _notify(f'<b style="color:#ad2330">File import failed: {escaped(exc)}</b>')

uploader.observe(on_upload_change,names='value')
load_uploaded_button.on_click(on_upload_fallback)

def logger(msg):
    with log_output:
        print(f'[{time.strftime("%H:%M:%S")}] {msg}',flush=True)

def on_generate(_):
    frame=state.get('frame')
    if frame is None:
        _notify('<b style="color:#ad2330">No compound CSV found. Please upload your CSV in Cell 3 first.</b>')
        return
    selected=[f for on,f in [(select76.value,'76'),(select_rd.value,'rdkit'),(select_fp.value,'ecfp')] if on]
    if not selected:
        _notify('<b style="color:#ad2330">Select at least one descriptor family.</b>')
        return
    if name_select.value==smiles_select.value:
        _notify('<b style="color:#ad2330">Select different compound-name and SMILES columns.</b>')
        return
    if not str(output_prefix.value).strip():
        _notify('<b style="color:#ad2330">Enter an output filename prefix.</b>')
        return
    run_btn.disabled=True
    progressbar.value=0;progressbar.bar_style='info'
    progress_text.value=f'<b style="color:#173956">0 / {len(frame):,} compounds</b>'
    with log_output:log_output.clear_output(wait=True)
    with summary_out:summary_out.clear_output(wait=True)
    _notify('<b style="color:#18579b">Descriptor generation in progress...</b>')
    base=Path('/content') if Path('/content').exists() else Path.cwd()
    folder=base/('QSAR_Part2_'+safe_prefix(output_prefix.value)+'_'+time.strftime('%Y%m%d_%H%M%S'))
    started=time.monotonic()
    def update_progress(n,total):
        pct=int(100*n/total) if total else 100
        progressbar.value=pct
        progress_text.value=(f'<b style="color:#173956">{n:,} / {total:,} compounds · '
                             f'{pct}% · elapsed {time.monotonic()-started:.0f}s</b>')
    try:
        logger(f'Input {state["filename"]}: {len(frame):,} rows')
        logger('Selected families: '+', '.join(selected))
        logger(f'RDKit {rdBase.rdkitVersion}: {len(RD_NAMES)} registry descriptors')
        result=generate_tables(frame,name_col=name_select.value,
                               smiles_col=smiles_select.value,
                               activity_col=activity_select.value or None,
                               selected=selected,output_root=folder,
                               prefix=output_prefix.value,qc_plots=qc_box.value,
                               deduplicate=dedup_box.value,
                               progress=update_progress,log=logger)
        state['result']=result
        progressbar.value=100;progressbar.bar_style='success'
        s=result['summary']
        _notify(f'<b style="color:#087966">Completed: {s["valid_rows"]:,} valid compounds and {s["invalid_rows"]:,} invalid SMILES. '
                f'Raw features {s["raw_feature_count"]:,}; cleaned features {s["cleaned_feature_count"]:,}.</b>')
        options=[('Complete ZIP',result['zip']),
                 ('Cleaned combined descriptor CSV',result['cleaned']),
                 ('Raw combined descriptor CSV',result['combined']),
                 ('Definition-duplicate audit CSV',result['audit'])]
        options += [(f'{k} descriptor-family CSV',v) for k,v in result['files'].items()]
        options += [('Methods.txt',str(folder/'Methods.txt')),
                    ('Results.txt',str(folder/'Results.txt')),
                    ('Summary.json',str(folder/'Summary.json')),
                    ('Invalid SMILES report',str(folder/f'{safe_prefix(output_prefix.value)}_Invalid_SMILES.csv'))]
        download_options.options=options
        with summary_out:
            display(HTML('<div style="color:#123e5d;font-size:14px;font-weight:800;padding:9px 0">'
                         f'Processed {s["valid_rows"]:,}/{s["input_rows"]:,} compounds · '
                         f'Known pIC50 {s["known_pIC50"]:,} · Unknown {s["unknown_pIC50"]:,} · '
                         f'Invalid SMILES {s["invalid_rows"]:,}</div>'))
            display(pd.DataFrame(result['preview']))
            display(FileLink(result['zip']))
    except Exception as exc:
        progressbar.bar_style='danger'
        logger('ERROR: '+repr(exc))
        logger(traceback.format_exc(limit=8))
        _notify(f'<b style="color:#ad2330">Analysis stopped: {escaped(type(exc).__name__)} — {escaped(exc)}</b>')
    finally:
        run_btn.disabled=False

def download_file(path):
    if not path or not os.path.isfile(path):
        with download_status:print('Output file is not available.')
        return
    try:
        from google.colab import files
        files.download(path)
    except Exception as exc:
        with download_status:
            print('Colab browser download unavailable:',exc)
            display(FileLink(path))

def on_download_selected(_):
    with download_status:download_status.clear_output(wait=True)
    download_file(download_options.value)

def on_download_zip(_):
    with download_status:download_status.clear_output(wait=True)
    if state['result'] is not None:
        download_file(state['result']['zip'])
    else:
        with download_status:print('Generate descriptors before downloading.')

run_btn.on_click(on_generate)
download_btn.on_click(on_download_selected)
download_zip_btn.on_click(on_download_zip)

upload_panel=panel(
    header('01 — Compound CSV upload','Use the native Colab file-picker in Cell 3. A backup CSV uploader is provided here if needed.'),
    widgets.HBox([uploader,load_uploaded_button],layout=widgets.Layout(flex_flow='row wrap',align_items='center')),
    status,preview_out
)
column_panel=panel(
    header('02 — Identify CSV columns','Choose the columns containing compound names, original SMILES and optional pIC50 values.'),
    widgets.HBox([field('Compound name',name_select),
                   field('SMILES',smiles_select),
                   field('pIC50 (optional)',activity_select)],
                  layout=widgets.Layout(flex_flow='row wrap'))
)
feature_panel=panel(
    header('03 — Select descriptor families','Select one, two or all three families. RDKit descriptors are counted from the installed version.'),
    checkline(select76,'Original 76 molecular descriptors','Includes 10 conformer-based 3D values'),
    checkline(select_rd,f'Complete RDKit registry — {len(RD_NAMES)} descriptors',f'RDKit {rdBase.rdkitVersion}'),
    checkline(select_fp,'ECFP4 fingerprint','Morgan radius 2 · 2048 binary bits'),
    checkline(dedup_box,'Export definition-deduplicated combined CSV','Raw original-family CSV files are also retained'),
    checkline(qc_box,'Generate sampled QC plots','High-resolution TIFF, 600 DPI'),
    widgets.HBox([field('Output filename prefix',output_prefix),run_btn],layout=widgets.Layout(flex_flow='row wrap',align_items='flex-end'))
)
progress_panel=panel(
    header('04 — Progress and data quality','Every input compound is counted, including invalid SMILES.'),
    progressbar,progress_text,log_output,summary_out
)
export_panel=panel(
    header('05 — Download analysis results','Download an individual CSV or the complete ZIP after calculation.'),
    widgets.HBox([field('Choose an output file',download_options),download_btn,download_zip_btn],
                  layout=widgets.Layout(flex_flow='row wrap',align_items='flex-end')),
    download_status
)
app=widgets.VBox([upload_panel,column_panel,feature_panel,progress_panel,export_panel],
                 layout=widgets.Layout(width='100%',max_width='1150px',gap='6px'))
app.add_class('qs-app')
display(app)

# Use the native-uploaded DataFrame immediately; no additional path or click necessary.
try:
    if isinstance(QSAR_INPUT_DF,pd.DataFrame) and not QSAR_INPUT_DF.empty:
        _load_dataframe(QSAR_INPUT_DF, QSAR_INPUT_NAME or 'uploaded.csv')
except NameError:
    pass
except Exception as exc:
    _notify(f'<b style="color:#ad2330">Previously uploaded CSV could not be loaded: {escaped(exc)}</b>')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Quick User Guide

1. Open this notebook using the **Google Colab website**, select **Runtime → Run all**, and grant permissions to upload the file from your computer.
2. In **Cell 3**, use Google's built-in upload prompt to choose a `.csv`. **Do not type any file path.** The imported filename and data preview are shown immediately.
3. **Cell 4** displays the descriptor studio in five white, high-contrast panels. Check the auto-detected `compound_name`, `smiles` and optional `pIC50` columns.
4. Select any combination of **76 custom descriptors**, **complete RDKit descriptors** and **ECFP4 fingerprint (2048 bits)**. Enable definition deduplication if a cleaned combined CSV is desired; original feature-family CSV files are preserved.
5. Click **Generate Selected Descriptors**. Monitor the progress and review valid/invalid compound counts.
6. Choose a file and click **Download Selected**, or download all results as ZIP.
7. To upload a different CSV, rerun **Cell 3 and Cell 4**, or try the backup uploader inside the dashboard.

### Scientific safeguards

- Original names, original SMILES and optional pIC50 are retained.
- The legacy `LogD` is `NaN` (not misleadingly replaced with logP).
- Original 76-feature CSV always has 76 named feature columns, including optional missing LogD.
- Complete RDKit descriptor count is detected from installed RDKit version rather than hardcoded.
- ECFP4 is Morgan radius 2, **2048 bits**, not a separate fourth fingerprint.
- No dataset-dependent variance or correlation feature selection occurs here. Perform this only within training folds during QSAR modeling.
- The 76 custom set includes ten slower 3D descriptors. 3D descriptor failures are recorded as missing values.

### Notes

The upload cell uses Google's officially documented `google.colab.files.upload()`; the older optional local-path field has been removed entirely. Depending on your browser/session the fallback ipywidgets uploader may behave differently, which is why the native Colab uploader is the primary method.
